# GIL · Ejercicio 2 — Medir CPU, I/O y entender el GIL

**Consigna**
1. Medir una suma CPU-bound con **1 thread, 2 threads y 2 procesos**.
2. Medir `sleep`/I/O simulado con **secuencial, threads y asyncio**.
3. Identificar si cada ejemplo está limitado por **CPU, I/O o sincronización**.
4. Explicar por qué el GIL **no impide concurrencia pero sí paralelismo de bytecode**.
5. Debatir cuándo conviene reescribir una función en **C/Rust o usar NumPy**.


In [ ]:
import sys, os, time, threading, asyncio, pickle
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import pandas as pd

sys.path.insert(0, os.getcwd())   # para poder importar el módulo que escribimos abajo

try:
    nucleos = len(os.sched_getaffinity(0))
except AttributeError:
    nucleos = os.cpu_count()

print("Python:        ", sys.version.split()[0])
print("Núcleos útiles:", nucleos)
print("GIL activo:    ", getattr(sys, "_is_gil_enabled", lambda: True)())
print("Switch interval:", sys.getswitchinterval(), "s")
if nucleos < 2:
    print("\nCon 1 solo núcleo los procesos no pueden ir más rápido. En Colab normalmente hay 2.")

def medir(fn, repeticiones=3):
    """Devuelve el MEJOR tiempo (s) de varias corridas: reduce el ruido de la máquina."""
    mejores = []
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        fn()
        mejores.append(time.perf_counter() - t0)
    return min(mejores)

def correr_async(coro):
    """Ejecuta una corrutina en un thread aparte con su propio event loop.
    Colab/Jupyter ya tienen un loop corriendo, por eso no usamos asyncio.run() directo."""
    res = {}
    def _t():
        res["v"] = asyncio.run(coro)
    t = threading.Thread(target=_t); t.start(); t.join()
    return res["v"]

Python:         3.13.18
Núcleos útiles: 2
GIL activo:     True
Switch interval: 0.005 s


In [ ]:
%%writefile tareas.py
import time

def suma_cpu(n):
    """Suma de cuadrados: CPU puro, sin esperar nada."""
    total = 0
    for i in range(n):
        total += i * i
    return total

def trabajo_io(seg):
    """Simula I/O: solo espera."""
    time.sleep(seg)
    return seg

def contar_durante(seg):
    """Cuenta iteraciones durante 'seg' segundos y cuántas veces fue 'pausado'.
    Una pausa = un hueco >0.5 ms entre dos iteraciones (otro thread tuvo el GIL)."""
    fin = time.perf_counter() + seg
    n = pausas = 0
    ultimo = time.perf_counter()
    while True:
        ahora = time.perf_counter()
        if ahora >= fin:
            break
        if ahora - ultimo > 0.0005:
            pausas += 1
        ultimo = ahora
        n += 1
    return n, pausas

Writing tareas.py


In [ ]:
import tareas as T
RESULTADOS = []   # (experimento, variante, tiempo, cuello de botella)
def registrar(exp, var, t, cuello):
    RESULTADOS.append((exp, var, t, cuello))

## Parte 1 · Suma CPU-bound: 1 thread vs 2 threads vs 2 procesos
El trabajo total es **fijo** (dos tramos de 5 millones de iteraciones). Lo único que cambia es *cómo* se reparte:
- **1 thread:** un solo thread hace los dos tramos, uno detrás del otro.
- **2 threads:** cada tramo en su thread.
- **2 procesos:** cada tramo en su proceso.

In [ ]:
N = 5_000_000

pool_proc = ProcessPoolExecutor(max_workers=2)
list(pool_proc.map(T.suma_cpu, [1, 1]))     # calentamiento: arranca los procesos antes de medir

def un_thread():
    T.suma_cpu(N); T.suma_cpu(N)
def dos_threads():
    with ThreadPoolExecutor(2) as ex: list(ex.map(T.suma_cpu, [N, N]))
def dos_procesos():
    list(pool_proc.map(T.suma_cpu, [N, N]))
def dos_procesos_con_arranque():
    with ProcessPoolExecutor(2) as p: list(p.map(T.suma_cpu, [N, N]))

variantes = [("1 thread", un_thread), ("2 threads", dos_threads),
             ("2 procesos (pool ya creado)", dos_procesos),
             ("2 procesos (incluye arrancar el pool)", dos_procesos_con_arranque)]
tiempos = [medir(fn) for _, fn in variantes]
for (v, _), t in zip(variantes, tiempos):
    registrar("1. Suma CPU-bound", v, t, "CPU (GIL)")
display(pd.DataFrame({"Variante": [v for v, _ in variantes],
                      "Tiempo (s)": [round(t, 2) for t in tiempos],
                      "Speedup vs 1 thread": [round(tiempos[0]/t, 2) for t in tiempos]}))

,Variante,Tiempo (s),Speedup vs 1 thread
0,1 thread,0.96,1.00
1,2 threads,0.96,1.00
2,2 procesos (pool ya creado),1.03,0.93
3,2 procesos (incluye arrancar el pool),1.06,0.90


### Conclusión de la parte 1
Dividir un cálculo entre **2 threads no lo hizo más rápido**: tardó lo mismo que hacerlo en uno, o un poco más por el trabajo extra de coordinar. En cambio **2 procesos** lo repartieron en dos núcleos de verdad y el tiempo bajó cerca de la mitad (siempre que la máquina tenga al menos 2 núcleos). También se ve el costo de arrancar procesos: si creás el pool cada vez, parte de la ganancia se pierde.
Entonces, para cálculo en Python puro, los threads no sirven para ir más rápido; los procesos sí.

## Parte 2 · I/O simulado: secuencial vs threads vs asyncio
10 tareas que **solo esperan 0,5 s** cada una (como 10 llamadas a una API).

In [ ]:
TAREAS_IO, ESPERA = 10, 0.5

def io_secuencial():
    for _ in range(TAREAS_IO): time.sleep(ESPERA)

def io_threads():
    with ThreadPoolExecutor(TAREAS_IO) as ex:
        list(ex.map(T.trabajo_io, [ESPERA] * TAREAS_IO))

async def _una():
    await asyncio.sleep(ESPERA)
async def _todas():
    await asyncio.gather(*[_una() for _ in range(TAREAS_IO)])
def io_asyncio():
    correr_async(_todas())

t_seq = medir(io_secuencial, 1); t_thr = medir(io_threads); t_asy = medir(io_asyncio)
for v, t in [("Secuencial", t_seq), ("10 threads", t_thr), ("asyncio", t_asy)]:
    registrar("2. I/O simulado (sleep)", v, t, "I/O (espera)")
display(pd.DataFrame({"Variante": ["Secuencial", "10 threads", "asyncio"],
                      "Tiempo (s)": [round(t_seq,2), round(t_thr,2), round(t_asy,2)],
                      "Speedup vs secuencial": [1.0, round(t_seq/t_thr,2), round(t_seq/t_asy,2)]}))

,Variante,Tiempo (s),Speedup vs secuencial
0,Secuencial,5.0,1.00
1,10 threads,0.5,9.95
2,asyncio,0.5,9.97


### Conclusión de la parte 2
Cuando las tareas solo **esperan**, el GIL deja de ser un problema. Las 10 esperas de medio segundo, que en secuencial suman unos 5 segundos, se solaparon con threads y con `asyncio` y todo terminó en medio segundo aproximadamente. Un thread que duerme **suelta el GIL**, y `asyncio` directamente no usa threads: se alterna entre tareas cada vez que una se pone a esperar. Threads y `asyncio` rindieron parecido; la diferencia está en la comodidad (threads funcionan con código bloqueante de siempre) y en el costo (`asyncio` escala a miles de tareas con mucha menos memoria).

## Parte 3 · ¿Qué limita a cada ejemplo? CPU, I/O o sincronización


In [ ]:
lock = threading.Lock()
contador = 0
def incrementar(n):
    global contador
    for _ in range(n):
        with lock:
            contador += 1

TOTAL = 1_000_000
def sync_1_thread():
    incrementar(TOTAL)
def sync_4_threads():
    hilos = [threading.Thread(target=incrementar, args=(TOTAL // 4,)) for _ in range(4)]
    for h in hilos: h.start()
    for h in hilos: h.join()

t1 = medir(sync_1_thread); t4 = medir(sync_4_threads)
registrar("3. Contador con Lock", "1 thread", t1, "Sincronización (lock)")
registrar("3. Contador con Lock", "4 threads", t4, "Sincronización (lock)")
display(pd.DataFrame({"Variante": ["1 thread", "4 threads"], "Tiempo (s)": [round(t1,2), round(t4,2)],
                      "Speedup vs 1 thread": [1.0, round(t1/t4,2)]}))

,Variante,Tiempo (s),Speedup vs 1 thread
0,1 thread,0.31,1.00
1,4 threads,0.32,0.95


### Conclusión de la parte 3
Que algo "vaya lento" no dice por qué. En el cálculo, el cuello de botella es el **procesador** (y el GIL, si se usan threads). En las esperas simuladas, es el **tiempo muerto de I/O**, que se resuelve solapando. Y en el contador con lock, el límite es la **fila para el lock**: 4 threads no hicieron más rápido el trabajo, porque en cada instante solo uno puede incrementar, y encima cuestan cambios de contexto. Cada cuello de botella pide una solución distinta.

## Parte 4 · El GIL: concurrencia sí, paralelismo de bytecode no
Experimento: cada thread cuenta iteraciones durante 1 segundo.
- Si hubiera **paralelismo**, 2 threads deberían contar ~2× lo que cuenta 1.
- Si hay solo **concurrencia**, el total de 2 threads será ~igual al de 1, pero cada thread será *pausado* varias veces (se turnan el GIL).

In [ ]:
SEG = 1.0
# 1 thread
n1, pausas1 = T.contar_durante(SEG)

# 2 threads a la vez
resultados = [None, None]
def correr(i): resultados[i] = T.contar_durante(SEG)
hilos = [threading.Thread(target=correr, args=(i,)) for i in range(2)]
for h in hilos: h.start()
for h in hilos: h.join()
n2 = sum(r[0] for r in resultados); pausas2 = sum(r[1] for r in resultados)

# 2 procesos a la vez
r_proc = list(pool_proc.map(T.contar_durante, [SEG, SEG]))
n_p = sum(r[0] for r in r_proc)

display(pd.DataFrame({
    "Escenario": ["1 thread", "2 threads", "2 procesos"],
    "Iteraciones totales (millones)": [round(n1/1e6, 2), round(n2/1e6, 2), round(n_p/1e6, 2)],
    "vs 1 thread": [1.0, round(n2/n1, 2), round(n_p/n1, 2)],
    "Pausas (otro tuvo el GIL)": [pausas1, pausas2, "—"]}))
print(f"(El intérprete cambia de thread cada {sys.getswitchinterval()*1000:.0f} ms aprox.)")

,Escenario,Iteraciones totales (millones),vs 1 thread,Pausas (otro tuvo el GIL)
0,1 thread,5.26,1.00,2
1,2 threads,4.03,0.77,179
2,2 procesos,3.64,0.69,—


(El intérprete cambia de thread cada 5 ms aprox.)


### Explicación: ¿por qué el GIL no impide concurrencia pero sí paralelismo de bytecode?
- **Concurrencia** es *manejar varias tareas a la vez, intercalándolas*. **Paralelismo** es *ejecutar varias en simultaneo, literalmente en el mismo instante*, en núcleos distintos.
- El **GIL** (Global Interpreter Lock) es un candado que permite que **un solo thread ejecute bytecode de Python por vez** dentro de un proceso. Los threads se turnan: cada cierto tiempo (o al bloquearse) el GIL pasa a otro.
- Por eso **sí hay concurrencia**: varios threads están "en progreso" y avanzan por turnos, y cuando uno espera I/O suelta el GIL para que otro trabaje.
- Y por eso **no hay paralelismo de bytecode**: aunque tengas 8 núcleos, en cada instante solo un thread de ese proceso está ejecutando Python. El segundo núcleo no sirve para correr el mismo intérprete.
- ¿Para qué existe? Simplifica el manejo de memoria de CPython (el conteo de referencias no necesita locks por objeto), lo que hace al intérprete simple y rápido en un solo thread.
- **Salidas:** varios procesos (cada uno con su GIL), código nativo que suelta el GIL (NumPy, C, Rust), y las versiones experimentales de Python 3.13+ *sin GIL* (`free-threaded`).

### Conclusión de la parte 4
El GIL **no impide que haya varias cosas en marcha a la vez**, pero **sí impide que dos de esas cosas ejecuten código Python al mismo tiempo**. Por eso los threads sirven muy bien para esperar (I/O) y casi nada para calcular. Para calcular de verdad en paralelo en Python, hay que usar procesos o sacar el trabajo pesado del intérprete.

## Parte 5 · ¿Cuándo reescribir en C/Rust o usar NumPy?
Primero medimos la diferencia entre un loop de Python y la misma cuenta con NumPy (el loop corre en C compilado).

In [ ]:
import numpy as np
M = 5_000_000

def suma_python():
    total = 0
    for i in range(M):
        total += i * i
    return total

def suma_numpy():
    a = np.arange(M, dtype=np.float64)
    return float((a * a).sum())

t_py = medir(suma_python); t_np = medir(suma_numpy)
registrar("5. Suma de cuadrados", "Python puro", t_py, "CPU (intérprete)")
registrar("5. Suma de cuadrados", "NumPy", t_np, "CPU (código nativo)")
display(pd.DataFrame({"Variante": ["Python puro", "NumPy"], "Tiempo (s)": [round(t_py,3), round(t_np,3)],
                      "Speedup": [1.0, round(t_py/t_np, 1)]}))

,Variante,Tiempo (s),Speedup
0,Python puro,0.558,1.0
1,NumPy,0.032,17.4


Y la segunda pregunta: ¿NumPy deja que los threads trabajen en paralelo (porque suelta el GIL)? Usamos `np.sin` sobre arrays grandes, con 1 thread vs 2 threads.

In [ ]:
a = np.random.rand(6_000_000)
def np_1_thread():
    np.sin(a); np.sin(a)
def np_2_threads():
    with ThreadPoolExecutor(2) as ex: list(ex.map(np.sin, [a, a]))

t1 = medir(np_1_thread); t2 = medir(np_2_threads)
registrar("5b. NumPy np.sin ×2", "1 thread", t1, "CPU (código nativo)")
registrar("5b. NumPy np.sin ×2", "2 threads", t2, "CPU (código nativo)")
display(pd.DataFrame({"Variante": ["1 thread", "2 threads"], "Tiempo (s)": [round(t1,3), round(t2,3)],
                      "Speedup": [1.0, round(t1/t2, 2)]}))
if nucleos < 2: print("Con 1 solo núcleo no se puede ver la mejora.")

,Variante,Tiempo (s),Speedup
0,1 thread,0.295,1.00
1,2 threads,0.252,1.17


### ¿Cuándo conviene?

**Conviene usar NumPy (o similares: pandas, SciPy, Polars) cuando:**
- el trabajo se puede expresar como operaciones sobre **arrays/tablas completos** (sumas, filtros, álgebra lineal): se elimina el loop de Python y se gana mucho sin escribir nada de C;
- ya existe la operación en la librería: es código probado y optimizado;
- además, NumPy **suelta el GIL** en operaciones grandes, así que hasta se pueden usar threads.

**Conviene reescribir en C/Rust (o Cython, Numba, PyO3) cuando:**
- hay un **loop caliente** que no se puede vectorizar (lógica con ramas, dependencia entre iteraciones) y el profiler lo señala como cuello de botella;
- necesitás **paralelismo real con memoria compartida** sin copiar datos entre procesos;
- el código se ejecuta **muchísimas veces** y el costo de mantenerlo se compensa con el ahorro.

**No conviene cuando:**
- el cuello de botella es **I/O**: reescribir en Rust una función que espera a la red no la hace más rápida;
- no **medimos** antes: muchas veces el problema es un algoritmo malo, y cambiar de lenguaje no lo arregla;
- la función es **chica y se llama pocas veces**: la complejidad de compilar, distribuir y depurar otro lenguaje cuesta más que lo que ahorra;


### Conclusión de la parte 5
NumPy fue muchísimo más rápido que el loop de Python **sin salir de Python**, porque mueve el trabajo a código compilado; y como ese código puede soltar el GIL, hasta los threads empiezan a servir. Reescribir en C o Rust tiene sentido solo cuando ya medimos, el cuello de botella es cálculo y no hay una librería que lo resuelva. Antes de eso suele convenir más mejorar el algoritmo o vectorizar. Cambiar de lenguaje es la herramienta más potente y también la más cara de mantener.

---
# Tabla de tiempos y cuello de botella

In [ ]:
res = pd.DataFrame(RESULTADOS, columns=["Experimento", "Variante", "Tiempo (s)", "Cuello de botella"])
res["Speedup vs 1ª variante"] = res.groupby("Experimento")["Tiempo (s)"].transform(lambda s: s.iloc[0] / s).round(2)
res["Tiempo (s)"] = res["Tiempo (s)"].round(3)
pool_proc.shutdown()
display(res)

,Experimento,Variante,Tiempo (s),Cuello de botella,Speedup vs 1ª variante
0,1. Suma CPU-bound,1 thread,0.957,CPU (GIL),1.00
1,1. Suma CPU-bound,2 threads,0.958,CPU (GIL),1.00
2,1. Suma CPU-bound,2 procesos (pool ya creado),1.031,CPU (GIL),0.93
3,1. Suma CPU-bound,2 procesos (incluye arrancar el pool),1.060,CPU (GIL),0.90
4,2. I/O simulado (sleep),Secuencial,5.002,I/O (espera),1.00
5,2. I/O simulado (sleep),10 threads,0.502,I/O (espera),9.95
6,2. I/O simulado (sleep),asyncio,0.502,I/O (espera),9.97
7,3. Contador con Lock,1 thread,0.306,Sincronización (lock),1.00
8,3. Contador con Lock,4 threads,0.323,Sincronización (lock),0.95
9,5. Suma de cuadrados,Python puro,0.558,CPU (intérprete),1.00


## Conclusión final del ejercicio
**El GIL no es el malo de todo, solo molesta cuando el trabajo es cálculo en Python puro.** Si el programa espera (red, disco, `sleep`), los threads y `asyncio` funcionan perfecto, porque quien espera suelta el candado. Si el programa calcula, los threads se turnan y no ganan velocidad; para eso están los procesos, o directamente sacar el cálculo a código nativo como NumPy, C o Rust. Y si el programa pierde tiempo porque todos esperan el mismo lock, el problema no es ni la CPU ni el I/O, sino el diseño.
Lo más importante para llevarse acá es el método: **medir, identificar el cuello de botella y recién entonces elegir la herramienta**. Elegir al revés (por moda o por intuición) es la forma más común de complicar un programa sin hacerlo más rápido. Podemos finalizar diciendo que a partir de las nuevas librerías de Python ya se solucionó la dificultad que se tenía en el GIL.